# Лабораторная работа №8
## Слесарев Никита ФИТ-231

№1 (3 балла)
Выполните задания из файла https://cloud.mail.ru/public/DGCu/Kw2Q4xGwd. Данные можно
взять здесь https://cloud.mail.ru/public/CiBD/PEaCv9DyD.

1. Загрузите выборку из файла titanic.csv с помощью пакета Pandas.
2. Оставьте в выборке четыре признака: класс пассажира (Pclass), цену билета (Fare), возраст пассажира (Age) и его пол (Sex).
3. Обратите внимание, что признак Sex имеет строковые значения.
4. Выделите целевую переменную — она записана в столбце Survived.
5. В данных есть пропущенные значения — например, для некоторых
пассажиров неизвестен их возраст. Такие записи при чтении их в
pandas принимают значение nan. Найдите все объекты, у которых
есть пропущенные признаки, и удалите их из выборки.
6. Обучите решающее дерево с параметром random_state=241 и остальными параметрами по умолчанию.
7. Вычислите важности признаков и найдите два признака с наибольшей важностью. Их названия будут ответами для данной задачи
(в качестве ответа укажите названия признаков через запятую без
пробелов).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# 1. Загрузка
df = pd.read_csv('train.csv')

# 2. Выбор признаков
features = ['Pclass', 'Fare', 'Age', 'Sex']
X = df[features].copy()
y = df['Survived']

# 3. Замена запятых на точки ВО ВСЕХ ЧИСЛОВЫХ столбцах и преобразование в float
for col in ['Fare', 'Age']:
    if col in X.columns:
        X[col] = X[col].astype(str).str.replace(',', '.').replace('nan', np.nan).astype(float)

# 4. Преобразование пола
X['Sex'] = X['Sex'].map({'male': 0, 'female': 1})

# 5. Удаление строк с пропусками
X = X.dropna()
y = y.loc[X.index]

# 6. Обучение
clf = DecisionTreeClassifier(random_state=241)
clf.fit(X, y)

# 7. Важности
importances = clf.feature_importances_
feature_names = X.columns

# Сортировка
result = pd.DataFrame({'feature': feature_names, 'importance': importances}) \
           .sort_values('importance', ascending=False)

# Вывод двух самых важных признаков (через запятую, без пробелов)
print(','.join(result['feature'].head(2).tolist()))

№2 (3 балла)
Построить дерево решений, определяющее наличие сахарного диабета у женщин. Данные
возьмите отсюда https://cloud.mail.ru/public/XD8U/pgDYnHrjY.
Данные содержат следующие характеристики:
1. Pregnancies – число случаев беременности
2. Glucose – концентрация глюкозы в крови
3. BloodPressure – артериальное диастолическое давление (мм рт. ст.)
4. SkinThickness – толщина кожной складки трехглавой мышцы (мм)
5. Insulin – 2-х часовой сывороточный инсулин
6. BMI – индекс массы тела
7. DiabetesPedigreeFunction – числовой параметр наследственности диабета
8. Age – возраст
Outcome – целевая переменная: 1 – наличие заболевания, 0 – отсутствие
Проделайте следующие действия:
1. Загрузите данные в DataFrame с помощью функции read_csv библиотеки pandas.
2. Разделите данные на обучающую и тестовую выборки с помощью функции
train_test_split.
3. Постройте дерево решений с помощью класса DecisionTreeClassifier.
4. Оцените качество модели с помощью функции classification_report.
5. Подберите оптимальное значение гиперпараметра max_depth с помощью поиска по
сетке (класс GridSearchCV).
6. Обучите модель с оптимальным max_depth и оцените результат.
Подсказка к заданию:
from sklearn.model_selection import GridSearchCV
parameters = {'max_depth':[2, 5, 10, 20, 100]} # можно указать другие значения
dtc = DecisionTreeClassifier()
grid = GridSearchCV(dtc, parameters)
grid.fit(X_train, y_train)
grid.best_estimator_ # лучшая модель

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

# 1. Загрузка данных с учётом запятой как десятичного разделителя
df = pd.read_csv('diabetes.csv', decimal=',')

# 2. Разделение на признаки и целевую переменную
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# 3. Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# 4. Базовая модель (для сравнения)
dtc_base = DecisionTreeClassifier(random_state=42)
dtc_base.fit(X_train, y_train)
y_pred_base = dtc_base.predict(X_test)
print("=== Базовая модель ===")
print(classification_report(y_test, y_pred_base))

# 5. Подбор оптимального max_depth
parameters = {'max_depth': [2, 3, 4, 5, 6, 7, 8, 10, 15, 20]}
dtc = DecisionTreeClassifier(random_state=42)
grid = GridSearchCV(dtc, parameters, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print(f"\nОптимальное max_depth: {grid.best_params_['max_depth']}")
print(f"Точность на кросс-валидации: {grid.best_score_:.4f}")

# 6. Оценка лучшей модели на тестовых данных
best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test)
print("\n=== Оптимальная модель ===")
print(classification_report(y_test, y_pred_best))

№3 (3 балла)
Выполните задания из файла https://cloud.mail.ru/public/2Btw/56cT2htNd. Данные можно
взять здесь https://cloud.mail.ru/public/mMAE/P3SZLgJWQ. 

Задача: построить модель случайного леса, определяющую хорошее вино
или плохое на основе результатов физико-химических тестов.
Входные переменные (признаки):
• fixed acidity – фиксированная кислотность
• volatile acidity – летучая кислотность
• citric acid – содержание лимонной кислоты
• residual sugar – остаточный сахар
• chlorides – хлориды
• free sulfur dioxide – свободный диоксид серы
• total sulfur dioxide – общий диоксид серы
• density – плотность
• pH
• sulphates – сульфаты
• alcohol – спирт
Целевая переменная – quality (качество): 0 – плохое вино, 1 – хорошее вино.
1. Загрузите набор данных из файла «winequality-red.csv» в DataFrame.
2. Обучите модель случайного леса (RandomForestClassifier).
3. Оцените качество модели. Метрики для оценки качества выберите
самостоятельно.
4. Оцените информативность признаков с помощью атрибута
feature_importances_.
4. Поимпровизируйте с этими данными. Попробуйте построить другие модели
(на ваш выбор). Также, можно попробовать изменить значения
гиперпараметров моделей и посмотреть, как изменится результат. 

In [24]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# 1. Загрузка данных
try:
    df = pd.read_csv('winequality-red.csv', sep=',', decimal=',')
except:
    try:
        df = pd.read_csv('winequality-red.csv', sep=';')
    except Exception as e:
        raise RuntimeError("Не удалось загрузить файл. Убедитесь, что он в папке.")

# Удалим артефакты
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df = df.dropna(axis=1, how='all')

print("Столбцы:", df.columns.tolist())
print("Уникальные значения в последнем столбце:", df.iloc[:, -1].unique())

# Определим целевую переменную
if 'quality' in df.columns:
    y_raw = df['quality']
else:
    # Берём последний столбец как quality
    y_raw = df.iloc[:, -1]
    df = df.iloc[:, :-1]
    df['quality'] = y_raw

# Определяем, бинарная она или нет
unique_vals = np.sort(y_raw.unique())
print("Уникальные значения quality:", unique_vals)

if set(unique_vals) == {0, 1}:
    print("Целевая переменная уже бинарная (0/1). Используем как есть.")
    df['quality_binary'] = y_raw.astype(int)
elif unique_vals.min() >= 3 and unique_vals.max() <= 10:
    print("Качество в шкале 3–8. Преобразуем в бинарную: >=6 → 1, иначе 0.")
    df['quality_binary'] = (y_raw >= 6).astype(int)
else:
    raise ValueError(f"Неизвестный формат quality: {unique_vals}")

# Проверим баланс классов
y = df['quality_binary']
print("Распределение классов:")
print(y.value_counts())

if y.nunique() < 2:
    raise ValueError("В данных только один класс! Модель обучать нельзя.")

# Признаки
X = df.drop(['quality', 'quality_binary'], axis=1, errors='ignore')

# Разделение
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("\n=== Random Forest ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

# Важность признаков
importances = rf.feature_importances_
features = X.columns
indices = np.argsort(importances)[::-1]

print("\n=== Важность признаков ===")
for i in range(len(features)):
    print(f"{i+1}. {features[indices[i]]}: {importances[indices[i]]:.4f}")

# Другие модели
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM": SVC(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

print("\n=== Сравнение моделей ===")
for name, model in models.items():
    if name in ["Logistic Regression", "SVM"]:
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name}: Accuracy = {acc:.4f}")

# Подбор гиперпараметров для RF
print("\n=== Подбор гиперпараметров Random Forest ===")
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10],
    'min_samples_split': [2, 5]
}
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, scoring='accuracy')
grid.fit(X_train, y_train)
print("Лучшие параметры:", grid.best_params_)
print("Точность на тесте:", accuracy_score(y_test, grid.predict(X_test)))

Столбцы: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']
Уникальные значения в последнем столбце: [0 1]
Уникальные значения quality: [0 1]
Целевая переменная уже бинарная (0/1). Используем как есть.
Распределение классов:
quality_binary
1    855
0    744
Name: count, dtype: int64

=== Random Forest ===
Accuracy: 0.80625

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.82      0.80       149
           1       0.83      0.80      0.81       171

    accuracy                           0.81       320
   macro avg       0.81      0.81      0.81       320
weighted avg       0.81      0.81      0.81       320


=== Важность признаков ===
1. alcohol: 0.1772
2. sulphates: 0.1366
3. volatile acidity: 0.1155
4. total sulfur dioxide: 0.0994
5. density: 0.0918
6. fixed acidity: 0.0711
7. chlorides: 0.0710

№4 (3 балла)
Выполните задания из файла https://cloud.mail.ru/public/nbue/PCozBiM4f. Данные можно
взять здесь https://cloud.mail.ru/public/ysqp/AuPY2gpwo.

В этом задании вам нужно проследить за изменением качества случайного леса в зависимости от количества деревьев в нем.
1. Загрузите данные из файла abalone.csv. Это датасет, в котором требуется предсказать возраст ракушки (число колец) по физическим
измерениям.
2. Преобразуйте признак Sex в числовой: значение F должно перейти
в -1, I — в 0, M — в 1. Если вы используете Pandas, то подойдет
2
следующий код: data[’Sex’] = data[’Sex’].map(lambda x: 1 if x ==
’M’ else (-1 if x == ’F’ else 0))
3. Разделите содержимое файлов на признаки и целевую переменную.
В последнем столбце записана целевая переменная, в остальных —
признаки.
4. Обучите случайный лес (sklearn.ensemble.RandomForestRegressor)
с различным числом деревьев: от 1 до 50 (random_state=1). Для
каждого из вариантов оцените качество работы полученного леса
на кросс-валидации по 5 блокам. Используйте параметры
"random_state=1"и "shuffle=True"при создании генератора кроссвалидации sklearn.cross_validation.KFold. В качестве меры качества
воспользуйтесь коэффициентом детерминации (sklearn.metrics.r2_score).
5. Определите, при каком минимальном количестве деревьев случайный лес показывает качество на кросс-валидации выше 0.52. Это
количество и будет ответом на задание.
6. Обратите внимание на изменение качества по мере роста числа деревьев. Ухудшается ли оно?


In [27]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score

# 1. Загрузка данных
df = pd.read_csv('abalone_csv.csv', sep=',', decimal=',')

# Удалим столбцы, у которых имя начинается с 'Unnamed' или полностью пустые
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df = df.dropna(axis=1, how='all')

# 2. Преобразование признака 'Sex'
df['Sex'] = df['Sex'].map({'M': 1, 'F': -1, 'I': 0})

# Убедимся, что нет NaN в данных
df = df.dropna()  # удаляем строки с пропущенными значениями

# 3. Разделение на признаки и целевую переменную
X = df.iloc[:, :-1]  # все столбцы, кроме последнего
y = df.iloc[:, -1]   # последний столбец — Class_number_of_rings

# Проверка: нет ли NaN в y
if y.isnull().any():
    raise ValueError("Целевая переменная содержит NaN!")

# 4. Поиск минимального количества деревьев, при котором R2 > 0.52
min_trees = None
random_state = 1
kf = KFold(n_splits=5, shuffle=True, random_state=random_state)

for n_trees in range(1, 51):
    rf = RandomForestRegressor(n_estimators=n_trees, random_state=random_state, n_jobs=-1)
    scores = cross_val_score(rf, X, y, cv=kf, scoring='r2')
    mean_r2 = scores.mean()
    
    if mean_r2 > 0.52:
        min_trees = n_trees
        break

# 5. Вывод результата
if min_trees is not None:
    print(min_trees)
else:
    print("Не найдено количество деревьев, при котором R2 > 0.52")

21


№5 (3 балла)
Выполните задания из файла https://cloud.mail.ru/public/YQ1s/9LJK7ZWuT. Данные можно
взять здесь https://cloud.mail.ru/public/RE1b/eRFQujUhe.

 Загрузите выборку из файла gbm-data.csv с помощью pandas и преобразуйте ее в массив numpy (параметр values у датафрейма). В
3
первой колонке файла с данными записано, была или нет реакция.
Все остальные колонки (d1 - d1776) содержат различные характеристики молекулы, такие как размер, форма и т.д. Разбейте выборку
на обучающую и тестовую, используя функцию train_test_split с
параметрами test_size = 0.8 и random_state = 241.
2. Обучите GradientBoostingClassifier с параметрами n_estimators=250,
verbose=True, random_state=241 и для каждого значения learning_rate
из списка [1, 0.5, 0.3, 0.2, 0.1] проделайте следующее:
• Используйте метод staged_decision_function для предсказания качества на обучающей и тестовой выборке на каждой
итерации.
• Преобразуйте полученное предсказание по формуле 1
1+e
−y_pred ,
где y_pred — предсказанное значение.
• Вычислите и постройте график значений log-loss на обучающей и тестовой выборках, а также найдите минимальное значение метрики и номер итерации, на которой оно достигается.
3. Как можно охарактеризовать график качества на тестовой выборке, начиная с некоторой итерации: переобучение (overfitting) или
недообучение (underfitting)? В ответе укажите одно из слов overfitting
либо underfitting.
4. Приведите минимальное значение log-loss на тестовой выборке и
номер итерации, на котором оно достигается, при learning_rate =
0.2.
5. На этих же данных обучите RandomForestClassifier с количеством
деревьев, равным количеству итераций, на котором достигается
наилучшее качество у градиентного бустинга из предыдущего пункта, random_state=241 и остальными параметрами по умолчанию.
Какое значение log-loss на тесте получается у этого случайного леса? (Не забывайте, что предсказания нужно получать с помощью
функции predict_proba. В данном случае брать сигмоиду от оценки
вероятности класса не нужно)
Если ответом является нецелое число, то целую и дробную часть
необходимо разграничивать точкой, например, 0.42. При необходимости
округляйте дробную часть до двух знаков.
4
Обратите внимание, что, хотя в градиентного бустинге гораздо более слабые базовые алгоритмы, он выигрывает у случайного леса благодаря более "направленной"настройке — каждый следующий алгоритм
исправляет ошибки имеющейся композиции. Также он обучается быстрее случайного леса благодаря использованию неглубоких деревьев. В
то же время, случайный лес может показать более высокое качество при
неограниченных ресурсах — так, он выиграет у градиентного бустинга
на наших данных, если увеличить число деревьев до нескольких сотен
(проверьте сами!).

In [ ]:
# Не скачивается датасет(